# Home Depot Product Search Relevance 


<img src='https://static.ticimax.com/uploads/images/urun-arastirmasi-nedir-nasil-yapilir-ba3a64.jpg'>


Bu çalışmada `Home Depot Product Search Relevance` yarışması için arama terimi ile ürün bilgileri arasındaki uyumu tahmin eden bir başlangıç modeli oluşturulmuştur.


## Akış

1. Kütüphaneleri yükleme
2. Drive bağlama ve zip içeriğini tanımlama
3. Veri dosyalarını yükleme
4. Veri birleştirme
5. Metin temelli özellikler çıkarma
6. Model kurma
7. RMSE değerlendirmesi
8. Test tahmini ve submission
9. Sonuç


## 1. Kütüphaneleri Yükleme


In [1]:
# Bu bölümde arama uygunluğu tahmini için gerekli veri hazırlama ve modelleme kütüphanelerini içe aktarıyoruz.


In [2]:
from google.colab import drive
import os
import zipfile
import tempfile
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import hstack
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


## 2. Drive Bağlama ve Zip İçeriğini Tanımlama


In [ ]:
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/home-depot-product-search-relevance.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_members = zip_ref.namelist()

def find_zip_member(members, filename):
    for member in members:
        if member.endswith(filename):
            return member
    raise FileNotFoundError(f'{filename} zip içinde bulunamadi.')

train_member = find_zip_member(zip_members, 'train.csv.zip')
test_member = find_zip_member(zip_members, 'test.csv.zip')
product_member = find_zip_member(zip_members, 'product_descriptions.csv.zip')
attributes_member = find_zip_member(zip_members, 'attributes.csv.zip')
sample_submission_member = find_zip_member(zip_members, 'sample_submission.csv.zip')


## 3. Veri Dosyalarını Yükleme


In [6]:
def read_csv_from_nested_zip(zip_path, member_name, **kwargs):
    with tempfile.TemporaryDirectory() as tmpdir:
        inner_zip_path = os.path.join(tmpdir, os.path.basename(member_name))

        with zipfile.ZipFile(zip_path, 'r') as outer_zip:
            with outer_zip.open(member_name) as src_file, open(inner_zip_path, 'wb') as dst_file:
                dst_file.write(src_file.read())

        with zipfile.ZipFile(inner_zip_path, 'r') as inner_zip:
            inner_members = inner_zip.namelist()
            csv_member = next((m for m in inner_members if m.endswith('.csv')), None)
            if csv_member is None:
                raise FileNotFoundError('İç zip içinde csv bulunamadı.')
            with inner_zip.open(csv_member) as f:
                return pd.read_csv(f, **kwargs)

train = read_csv_from_nested_zip(zip_path, train_member, encoding='ISO-8859-1')
test = read_csv_from_nested_zip(zip_path, test_member, encoding='ISO-8859-1')
product_descriptions = read_csv_from_nested_zip(zip_path, product_member)
attributes = read_csv_from_nested_zip(zip_path, attributes_member)
sample_submission = read_csv_from_nested_zip(zip_path, sample_submission_member)

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Product descriptions shape:', product_descriptions.shape)
print('Attributes shape:', attributes.shape)
print('Sample submission shape:', sample_submission.shape)
train.head()


Train shape: (74067, 5)
Test shape: (166693, 4)
Product descriptions shape: (124428, 2)
Attributes shape: (2044803, 3)
Sample submission shape: (166693, 2)


,id,product_uid,product_title,search_term,relevance
0,2,100001,Simpson Strong-Tie 12-Gauge Angle,angle bracket,3.00
1,3,100001,Simpson Strong-Tie 12-Gauge Angle,l bracket,2.50
2,9,100002,BEHR Premium Textured DeckOver 1-gal. #SC-141 Tugboat Wood and Concrete Coating,deck over,3.00
3,16,100005,Delta Vero 1-Handle Shower Only Faucet Trim Kit in Chrome (Valve Not Included),rain shower head,2.33
4,17,100005,Delta Vero 1-Handle Shower Only Faucet Trim Kit in Chrome (Valve Not Included),shower only faucet,2.67


## 4. Veri Birleştirme


In [7]:
# Bu bölümde ürün açıklamalarını ve marka bilgisini train ve test tabloları ile birleştiriyoruz.


In [8]:
brand_df = attributes[attributes['name'] == 'MFG Brand Name'][['product_uid', 'value']].rename(columns={'value': 'brand'})

train_df = train.merge(product_descriptions, on='product_uid', how='left').merge(brand_df, on='product_uid', how='left')
test_df = test.merge(product_descriptions, on='product_uid', how='left').merge(brand_df, on='product_uid', how='left')

for df in [train_df, test_df]:
    df['product_title'] = df['product_title'].fillna('')
    df['search_term'] = df['search_term'].fillna('')
    df['product_description'] = df['product_description'].fillna('')
    df['brand'] = df['brand'].fillna('')

print('Train merged shape:', train_df.shape)
print('Test merged shape:', test_df.shape)
train_df.head()


Train merged shape: (74067, 7)
Test merged shape: (166693, 6)


,id,product_uid,product_title,search_term,relevance,product_description,brand
0,2,100001,Simpson Strong-Tie 12-Gauge Angle,angle bracket,3.00,"Not only do angles make joints stronger, they also provide more consistent, straight corners. Simpson Strong-Tie offers a wide variety of angles in various sizes and thicknesses to handle light-duty jobs or projects where a structural connection is needed. Some can be bent (skewed) to match the project. For outdoor projects or those where moisture is present, use our ZMAX zinc-coated connectors, which provide extra resistance against corrosion (look for a ""Z"" at the end of the model number).Versatile connector for various 90 connections and home repair projectsStronger than angled nailing or screw fastening aloneHelp ensure joints are consistently straight and strongDimensions: 3 in. x 3 in. x 1-1/2 in.Made from 12-Gauge steelGalvanized for extra corrosion resistanceInstall with 10d common nails or #9 x 1-1/2 in. Strong-Drive SD screws",Simpson Strong-Tie
1,3,100001,Simpson Strong-Tie 12-Gauge Angle,l bracket,2.50,"Not only do angles make joints stronger, they also provide more consistent, straight corners. Simpson Strong-Tie offers a wide variety of angles in various sizes and thicknesses to handle light-duty jobs or projects where a structural connection is needed. Some can be bent (skewed) to match the project. For outdoor projects or those where moisture is present, use our ZMAX zinc-coated connectors, which provide extra resistance against corrosion (look for a ""Z"" at the end of the model number).Versatile connector for various 90 connections and home repair projectsStronger than angled nailing or screw fastening aloneHelp ensure joints are consistently straight and strongDimensions: 3 in. x 3 in. x 1-1/2 in.Made from 12-Gauge steelGalvanized for extra corrosion resistanceInstall with 10d common nails or #9 x 1-1/2 in. Strong-Drive SD screws",Simpson Strong-Tie
2,9,100002,BEHR Premium Textured DeckOver 1-gal. #SC-141 Tugboat Wood and Concrete Coating,deck over,3.00,"BEHR Premium Textured DECKOVER is an innovative solid color coating. It will bring your old, weathered wood or concrete back to life. The advanced 100% acrylic resin formula creates a durable coating for your tired and worn out deck, rejuvenating to a whole new look. For the best results, be sure to properly prepare the surface using other applicable BEHR products displayed above.California residents: see&nbsp;Proposition 65 informationRevives wood and composite decks, railings, porches and boat docks, also great for concrete pool decks, patios and sidewalks100% acrylic solid color coatingResists cracking and peeling and conceals splinters and cracks up to 1/4 in.Provides a durable, mildew resistant finishCovers up to 75 sq. ft. in 2 coats per gallonCreates a textured, slip-resistant finishFor best results, prepare with the appropriate BEHR product for your wood or concrete surfaceActual paint colors may vary from on-screen and printer representationsColors available to be tinted in most storesOnline Price includes Paint Care fee in the following states: CA, CO, CT, ME, MN, OR, RI, VT",BEHR Premium Textured DeckOver
3,16,100005,Delta Vero 1-Handle Shower Only Faucet Trim Kit in Chrome (Valve Not Included),rain shower head,2.33,"Update your bathroom with the Delta Vero Single-Handle Shower Faucet Trim Kit in Chrome. It has a sleek, modern and minimalistic aesthetic. The MultiChoice universal valve keeps the water temperature within +/-3 degrees Fahrenheit to help prevent scalding.California residents: see&nbsp;Proposition 65 informationIncludes the trim kit only, the rough-in kit (R10000-UNBX) is sold separatelyIncludes the handleMaintains a balanced pressure of hot and cold water even when a valve is turned on or off elsewhere in the systemDue to WaterSense regulations in the state of New York, please confirm your shipping zip code is not restricted from use of items that do not meet WaterSense qualifications",Delta

## 5. Metin Temelli Özellikler Çıkarma


In [9]:
# Bu bölümde arama terimi ile ürün bilgileri arasındaki temel metin ilişkilerini sayısal hale getiriyoruz.


In [10]:
def text_overlap_ratio(search, text):
    search_words = set(str(search).lower().split())
    text_words = set(str(text).lower().split())
    if len(search_words) == 0:
        return 0.0
    return len(search_words & text_words) / len(search_words)

for df in [train_df, test_df]:
    df['search_len'] = df['search_term'].apply(lambda x: len(str(x).split()))
    df['title_len'] = df['product_title'].apply(lambda x: len(str(x).split()))
    df['desc_len'] = df['product_description'].apply(lambda x: len(str(x).split()))
    df['brand_len'] = df['brand'].apply(lambda x: len(str(x).split()))
    df['title_overlap'] = df.apply(lambda row: text_overlap_ratio(row['search_term'], row['product_title']), axis=1)
    df['desc_overlap'] = df.apply(lambda row: text_overlap_ratio(row['search_term'], row['product_description']), axis=1)
    df['brand_overlap'] = df.apply(lambda row: text_overlap_ratio(row['search_term'], row['brand']), axis=1)
    df['combined_text'] = (
        df['search_term'].astype(str) + ' ' +
        df['product_title'].astype(str) + ' ' +
        df['product_description'].astype(str) + ' ' +
        df['brand'].astype(str)
    )

feature_columns = [
    'combined_text', 'search_len', 'title_len', 'desc_len', 'brand_len',
    'title_overlap', 'desc_overlap', 'brand_overlap'
]

x = train_df[feature_columns].copy()
y = train_df['relevance'].copy()
test_x = test_df[feature_columns].copy()

x_train, x_valid, y_train, y_valid = train_test_split(
    x, y, test_size=0.2, random_state=42
)

print('x_train shape:', x_train.shape)
print('x_valid shape:', x_valid.shape)


x_train shape: (59253, 8)
x_valid shape: (14814, 8)


## 6. Model Kurma


In [11]:
# Bu bölümde metin ve sayısal özellikleri birleştirerek arama uygunluğu tahmin eden başlangıç modelini kuruyoruz.


In [12]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')

x_train_text = tfidf.fit_transform(x_train['combined_text'])
x_valid_text = tfidf.transform(x_valid['combined_text'])
test_x_text = tfidf.transform(test_x['combined_text'])

numeric_cols = ['search_len', 'title_len', 'desc_len', 'brand_len', 'title_overlap', 'desc_overlap', 'brand_overlap']

x_train_num = x_train[numeric_cols].fillna(0).values
x_valid_num = x_valid[numeric_cols].fillna(0).values
test_x_num = test_x[numeric_cols].fillna(0).values

x_train_final = hstack([x_train_text, x_train_num])
x_valid_final = hstack([x_valid_text, x_valid_num])
test_x_final = hstack([test_x_text, test_x_num])

model = Ridge(alpha=1.0)
model.fit(x_train_final, y_train)


Ridge()

## 7. RMSE Değerlendirmesi


In [13]:
# Bu bölümde modelin arama uygunluğu tahminindeki hata düzeyini RMSE ile ölçüyoruz.


In [14]:
valid_preds = model.predict(x_valid_final)
rmse = root_mean_squared_error(y_valid, valid_preds)
print('Validation RMSE:', round(rmse, 5))


Validation RMSE: 0.4969


## 8. Test Tahmini ve Submission


In [15]:
# Bu bölümde test verisi için uygunluk skorlarını üretip submission dosyasını oluşturuyoruz.


In [16]:
test_preds = model.predict(test_x_final)

submission = sample_submission.copy()
submission['relevance'] = test_preds

print('Submission shape:', submission.shape)
submission.head()


Submission shape: (166693, 2)


,id,relevance
0,1,2.402686
1,4,2.278123
2,5,2.464211
3,6,2.487371
4,7,2.394585


In [17]:
output_path = '/content/home_depot_submission.csv'
submission.to_csv(output_path, index=False)
print('Kaydedilen dosya:', output_path)


Kaydedilen dosya: /content/home_depot_submission.csv


## 9. Sonuç


Bu çalışmada Home Depot Product Search Relevance yarışması için arama terimi ile ürün bilgileri arasındaki metin ilişkileri kullanılarak arama uygunluğu tahmini yapan bir başlangıç modeli oluşturuldu. Elde edilen sonuçlara göre model validation verisi üzerinde 0.49690 RMSE değeri elde etti ve test verisi için uygunluk skorları üretildi.
